# Chapter 10 - Blip3o SFT Script

In [ ]:
# Install a BLIP3o-compatible stack.
# Run this cell ONCE in a fresh runtime, then RESTART the runtime before continuing.

import sys
import subprocess


def pip(*args):
    cmd = [sys.executable, "-m", "pip", *args]
    print(" ".join(cmd))
    return subprocess.run(cmd, check=False)


# Remove conflicting preinstalls / prior runs
pip("uninstall", "-y", "gcsfs", "transformers", "diffusers", "accelerate", "peft", "datasets", "fsspec")

# Packaging tools
pip("install", "-q", "-U", "pip", "setuptools", "wheel")

# Stable stack for this notebook
pip(
    "install", "-q",
    "tokenizers",
    "sentencepiece",
    "shortuuid",
    "transformers==4.51.3",
    "accelerate==0.34.2",
    "peft==0.15.2",
    "diffusers==0.32.2",
    "bitsandbytes",
    "pydantic",
    "markdown2[all]",
    "numpy",
    "scikit-learn",
    "requests",
    "uvicorn",
    "fastapi",
    "einops==0.8.1",
    "einops-exts==0.0.4",
    "timm>=0.6.13",
    "ftfy",
    "datasets==2.16.1",
    "fsspec==2023.10.0",
    "tabulate",
    "ninja",
    "qwen_vl_utils",
    "huggingface_hub",
    "wandb",
    "torchvision",
    "pillow",
)

print("\nInstall complete.")
print("RESTART the runtime now, then continue with the next cell.")

/usr/bin/python3 -m pip uninstall -y gcsfs transformers diffusers accelerate peft datasets fsspec
/usr/bin/python3 -m pip install -q -U pip setuptools wheel
/usr/bin/python3 -m pip install -q tokenizers sentencepiece shortuuid transformers==4.51.3 accelerate==0.34.2 peft==0.15.2 diffusers==0.32.2 bitsandbytes pydantic markdown2[all] numpy scikit-learn requests uvicorn fastapi einops==0.8.1 einops-exts==0.0.4 timm>=0.6.13 ftfy datasets==2.16.1 fsspec==2023.10.0 tabulate ninja qwen_vl_utils huggingface_hub wandb torchvision pillow

Install complete.
RESTART the runtime now, then continue with the next cell.


Here we import the core libraries and define our training configuration

In [ ]:
import os
import re
import gc
import json
import math
import sys
import copy
import torch

import transformers
import accelerate
import diffusers
import peft

from datasets import load_dataset
from diffusers import AutoencoderKL
from huggingface_hub import snapshot_download
from peft import LoraConfig, get_peft_model, PeftModel
from PIL import Image
from safetensors.torch import load_file
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)

print("Python      :", sys.version.split()[0])
print("transformers:", transformers.__version__, transformers.__file__)
print("diffusers   :", diffusers.__version__, diffusers.__file__)
print("accelerate  :", accelerate.__version__, accelerate.__file__)
print("peft        :", peft.__version__, peft.__file__)

MODEL_ID      = "orrzohar/BLIP3o-4B-v3-TEST"
DIFFUSION_ID  = "orrzohar/BLIP3o-4B-Diffusion-Decoder"
PROCESSOR_ID  = "Qwen/Qwen2.5-VL-3B-Instruct"
DATASET_ID    = "orrzohar/BLIP3o-Visual-Reasoning"
OUTPUT_DIR    = "./outputs_visual_jigsaw"

IGNORE_INDEX    = -100
IMAGE_TOKEN_IDX = 151667
IMAGE_SIZE      = 448
LATENT_QUERIES  = 64
MAX_LENGTH      = 4096

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python      : 3.12.12
transformers: 4.51.3 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
diffusers   : 0.32.2 /usr/local/lib/python3.12/dist-packages/diffusers/__init__.py
accelerate  : 0.34.2 /usr/local/lib/python3.12/dist-packages/accelerate/__init__.py
peft        : 0.15.2 /usr/local/lib/python3.12/dist-packages/peft/__init__.py
NOTE: in this transformers version, use eval_strategy=... (not evaluation_strategy=...).
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Dataset and Data Loading

The `VisualReasoningDataset` class prepares our training data by:

1. Processing the input: Takes a puzzle image and question, tokenizes them using the Qwen processor
2. Parsing the reasoning chain: The model's response contains interleaved text and `<image>` tokens. We split these apart and:
   - Text segments are tokenized normally
   - `<image>` tags are replaced with 64 latent query tokens (placeholders for generated images)
3. Building training labels: We mask the input prompt (the model shouldn't be trained to predict the question) and only supervise on the assistant's reasoning response


The `collate_fn` handles batching by padding sequences to equal length, this is necessary because different samples have different numbers of reasoning steps and images.

In [ ]:
class VisualReasoningDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.data = hf_dataset
        self.processor = processor
        self.tokenizer = getattr(processor, "tokenizer", processor)
        self.pad_token_id = self.tokenizer.pad_token_id or 0

        self.gen_transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711],
            ),
        ])
        self.blank_image = self.gen_transform(Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        problem_image = sample["problem_image"]

        reasoning_images = [
            self.gen_transform(sample[k])
            for k in ["reasoning_image_1", "reasoning_image_2", "reasoning_image_3", "reasoning_image_4"]
            if sample.get(k) is not None
        ]

        processed = self.processor(
            text=sample["prefix_text"],
            images=[problem_image],
            padding=False,
            return_tensors="pt",
        )

        prefix_ids = processed["input_ids"][0]

        assistant_text = sample["assistant_text"]
        segments = re.split(r"(<image>)", assistant_text)

        assistant_ids = []
        gen_images = []
        labels_parts = []

        latent_block = torch.full((LATENT_QUERIES,), IMAGE_TOKEN_IDX, dtype=torch.long)
        reasoning_queue = list(reasoning_images)

        for seg in segments:
            if not seg:
                continue
            if seg == "<image>" and reasoning_queue:
                assistant_ids.append(latent_block)
                gen_images.append(reasoning_queue.pop(0))
                labels_parts.append(("latent", LATENT_QUERIES))
            elif seg != "<image>" and seg.strip():
                ids = self.tokenizer(seg.strip(), add_special_tokens=False, return_tensors="pt")["input_ids"][0]
                assistant_ids.append(ids)
                labels_parts.append(("text", ids))

        eos = torch.tensor([self.tokenizer.eos_token_id], dtype=torch.long)
        assistant_ids.append(eos)

        assistant_ids = torch.cat(assistant_ids) if assistant_ids else torch.tensor([], dtype=torch.long)
        input_ids = torch.cat([prefix_ids, assistant_ids])
        attention_mask = torch.ones_like(input_ids)

        labels = [torch.full_like(prefix_ids, IGNORE_INDEX)]
        for kind, val in labels_parts:
            if kind == "text":
                labels.append(val.clone())
            else:
                labels.append(torch.full((val,), IMAGE_TOKEN_IDX, dtype=torch.long))
        labels.append(eos.clone())
        labels = torch.cat(labels)

        if len(input_ids) > MAX_LENGTH:
            input_ids = input_ids[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]

        gen_images = torch.stack(gen_images) if gen_images else self.blank_image.unsqueeze(0)

        result = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "i_s_pos": torch.tensor(len(prefix_ids), dtype=torch.long),
            "gen_images": gen_images,
            "num_reasoning_images": torch.tensor(len(reasoning_images), dtype=torch.long),
            "pad_token_id": torch.tensor(self.pad_token_id, dtype=torch.long),
        }
        if processed.get("pixel_values") is not None:
            result["pixel_values"] = processed["pixel_values"]
        if processed.get("image_grid_thw") is not None:
            result["image_grid_thw"] = processed["image_grid_thw"]
        return result


def collate_fn(features):
    pad_id = int(features[0]["pad_token_id"])
    batch = {
        "input_ids": pad_sequence([f["input_ids"] for f in features], batch_first=True, padding_value=pad_id),
        "attention_mask": pad_sequence([f["attention_mask"] for f in features], batch_first=True, padding_value=0),
        "labels": pad_sequence([f["labels"] for f in features], batch_first=True, padding_value=IGNORE_INDEX),
        "i_s_pos": torch.stack([f["i_s_pos"] for f in features]),
        "num_reasoning_images": torch.stack([f["num_reasoning_images"] for f in features]),
    }

    pv_list = [f["pixel_values"] for f in features if "pixel_values" in f]
    if pv_list:
        batch["pixel_values"] = torch.cat(pv_list, dim=0)

    grid_list = [f["image_grid_thw"] for f in features if "image_grid_thw" in f]
    if grid_list:
        batch["image_grid_thw"] = torch.cat(grid_list, dim=0)

    gen_list = [f["gen_images"] for f in features]
    max_n = max(t.shape[0] for t in gen_list)
    padded = []
    for t in gen_list:
        if t.shape[0] < max_n:
            pad = torch.zeros((max_n - t.shape[0],) + t.shape[1:], dtype=t.dtype)
            t = torch.cat([t, pad], dim=0)
        padded.append(t)

    # IMPORTANT: model.forward expects gen_images (plural), not gen_image.
    batch["gen_images"] = torch.stack(padded)
    return batch

## Inference Helper and troubleshooting functions

In [ ]:
from torch.amp import autocast


def move_batch_to_cuda(batch):
    return {k: (v.to("cuda") if torch.is_tensor(v) else v) for k, v in batch.items()}


def show_sample(tag, s):
    print("=" * 70)
    print(tag)
    print("=" * 70)
    print(f"Prompt:\n{s['prefix_text'][:500]}...")
    print(f"\nGround truth:\n{s['assistant_text'][:500]}...")


@torch.no_grad()
def run_inference_bf16(model, processor, sample, max_new_tokens=64):
    # Text generation is only a side-check for this notebook.
    # The primary evaluation is the custom forward-path loss comparison in the final section.
    model.eval()
    device = next(model.parameters()).device

    try:
        inputs = processor(
            text=sample["prefix_text"],
            images=[sample["problem_image"]],
            return_tensors="pt",
        )
    except Exception:
        inputs = processor(
            text=sample["prefix_text"],
            images=sample["problem_image"],
            return_tensors="pt",
        )

    inputs = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

    with autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    return processor.tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


@torch.no_grad()
def eval_losses(model, batch):
    model.eval()
    with autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model(**batch)
    return {
        "loss": float(out.loss) if getattr(out, "loss", None) is not None else None,
        "text_loss": float(out.text_loss) if getattr(out, "text_loss", None) is not None else None,
        "img_loss": float(out.img_loss) if getattr(out, "img_loss", None) is not None else None,
    }


def print_triplet(title, vals):
    print("=" * 70)
    print(title)
    print("=" * 70)
    for k, v in vals.items():
        print(f"{k:>10}: {v:.4f}" if v is not None else f"{k:>10}: None")


def get_trainable_state(model):
    return {
        n: p.detach().cpu().clone()
        for n, p in model.named_parameters()
        if p.requires_grad
    }


def load_trainable_state(model, state_dict):
    name_to_param = dict(model.named_parameters())
    with torch.no_grad():
        for n, tensor in state_dict.items():
            if n in name_to_param:
                name_to_param[n].copy_(tensor.to(device=name_to_param[n].device, dtype=name_to_param[n].dtype))


def save_initial_trainable_state(model, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "initial_trainable_state.pt")
    torch.save(get_trainable_state(model), path)
    print(f"Saved initial trainable state to {path}")

## Load Dataset and Model

Now we load everything from HuggingFace:

1. Processor: Handles tokenization and image preprocessing (from Qwen2.5-VL)
2. Dataset: Visual reasoning puzzles with step-by-step solutions
3. Model: BLIP3o-4B with its diffusion components:
   - Base LLM (Qwen-based)
   - VAE decoder (for image generation)
   - Generation vision tower (EVA-CLIP encoder)


In [ ]:
# Load processor + dataset
processor = AutoProcessor.from_pretrained(
    PROCESSOR_ID,
    trust_remote_code=True,
    use_fast=False,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

raw_dataset = load_dataset(DATASET_ID, split="train")
split = raw_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = VisualReasoningDataset(split["train"], processor)
eval_dataset  = VisualReasoningDataset(split["test"], processor)
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Manual shard loading + explicit embedding patch
local_dir = snapshot_download(repo_id=MODEL_ID)
index_file = os.path.join(local_dir, "model.safetensors.index.json")
with open(index_file, "r") as f:
    weight_map = json.load(f)["weight_map"]

state_dict = {}
for sf_file in sorted(set(weight_map.values())):
    sf_path = os.path.join(local_dir, sf_file)
    print(f"Loading shard: {os.path.basename(sf_path)}")
    state_dict.update(load_file(sf_path, device="cpu"))

config = AutoConfig.from_pretrained(local_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_config(
    config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

load_result = model.load_state_dict(state_dict, strict=False)
print("Missing keys   :", len(load_result.missing_keys), load_result.missing_keys[:20])
print("Unexpected keys:", len(load_result.unexpected_keys), load_result.unexpected_keys[:20])

# CRITICAL PATCH:
# Checkpoint stores embeddings under model.language_model.embed_tokens.weight
# while the live custom class uses model.embed_tokens.weight.
with torch.no_grad():
    src_embed_key = "model.language_model.embed_tokens.weight"
    if src_embed_key not in state_dict:
        raise KeyError(f"Missing checkpoint tensor: {src_embed_key}")

    src_embed = state_dict[src_embed_key]
    dst_embed = model.get_model().embed_tokens.weight
    if tuple(src_embed.shape) != tuple(dst_embed.shape):
        raise ValueError(
            f"Embedding shape mismatch: checkpoint {tuple(src_embed.shape)} vs model {tuple(dst_embed.shape)}"
        )
    dst_embed.copy_(src_embed.to(dtype=dst_embed.dtype))
    print("Patched model.embed_tokens.weight from checkpoint")

    if "lm_head.weight" in state_dict:
        src_lm = state_dict["lm_head.weight"]
        dst_lm = model.lm_head.weight
        if tuple(src_lm.shape) == tuple(dst_lm.shape):
            dst_lm.copy_(src_lm.to(dtype=dst_lm.dtype))
            print("Patched lm_head.weight from checkpoint")

print("embed_tokens norm:", float(model.get_model().embed_tokens.weight.float().norm()))
print("lm_head norm     :", float(model.lm_head.weight.float().norm()))

del state_dict
gc.collect()

model = model.to("cuda")
model.config.use_cache = False

# Custom BLIP3o code combines total_loss = text_loss + img_loss by default.
# Starting with a smaller image-loss weight makes the training signal easier to read.
model.config.text_loss_weight = 1.0
model.config.img_loss_weight = 0.5
print("Loss weights  :", model.config.text_loss_weight, model.config.img_loss_weight)

model.get_model().vae = AutoencoderKL.from_pretrained(
    DIFFUSION_ID,
    subfolder="vae",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    variant="bf16",
).to("cuda")

gen_tower = model.get_gen_vision_tower()
gen_tower.load_model(device="cuda")
for m in gen_tower.modules():
    if hasattr(m, "xattn"):
        m.xattn = False

print("Model + dataset loaded.")

# Optional sanity smoke-test: text-only side-check.
sample = eval_dataset.data[0]
print("Text-only side check:", run_inference_bf16(model, processor, sample, max_new_tokens=16))

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00007-of-00009.safetensors:   0%|          | 0.00/2.06G [00:00<?, ?B/s]

model-00006-of-00009.safetensors:   0%|          | 0.00/2.12G [00:00<?, ?B/s]

model-00009-of-00009.safetensors:   0%|          | 0.00/764M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Model + dataset loaded.
Text-only side check: The


## Apply LoRA Adapters

Instead of fine-tuning all 4 billion parameters (expensive!), we use LoRA in this example to train a small percentage of the weights.

Those are the parameters that we are adjusting:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `r=8` | Rank of LoRA matrices | Higher = more capacity, more memory |
| `lora_alpha=16` | Scaling factor | Controls adapter contribution |
| `target_modules` | q/k/v/o projectors | Where adapters are inserted |
| `modules_to_save` | down projector |  |




In [ ]:
# Freeze base model and apply LIGHTER LoRA
model.requires_grad_(False)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    modules_to_save=["down_projector"],
)

model = get_peft_model(model, peft_config)

# latent_queries is a Parameter outside LoRA
model.get_model().latent_queries.requires_grad = True

# Helpful for PEFT / frozen-backbone training
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.config.use_cache = False
model.print_trainable_parameters()

trainable = []
frozen = 0
trainable_n = 0
for n, p in model.named_parameters():
    if p.requires_grad:
        trainable.append((n, p.numel()))
        trainable_n += p.numel()
    else:
        frozen += p.numel()

print("Trainable params:", trainable_n)
print("Frozen params   :", frozen)
print("\nTop trainable tensors:")
for n, k in sorted(trainable, key=lambda x: -x[1])[:30]:
    print(f"{k:>12}  {n}")

# Save the PRE-TRAIN trainable state for later base-vs-FT comparison
save_initial_trainable_state(model, OUTPUT_DIR)

trainable params: 12,210,176 || all params: 9,570,796,935 || trainable%: 0.1276
Trainable params: 12210176
Frozen params   : 9558586759

Top trainable tensors:
     4194304  base_model.model.model.down_projector.modules_to_save.default.0.weight
     4194304  base_model.model.model.down_projector.modules_to_save.default.2.weight
      131072  base_model.model.model.latent_queries
       16384  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
       16384  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
       16384  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
       16384  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
       16384  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
       16384  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
       16384  base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
       16384  base_mode

## Train

Time to fine-tune! We use HuggingFace's `Trainer` with the following settings:

| Setting | Value | Notes |
|---------|-------|-------|
| Epochs | 1 |   |
| Batch size | 1 | Per GPU |
| Gradient accumulation | 8 | Effective batch = 8 |
| Learning rate | 5e-6 | Standard for LoRA fine-tuning |
| Precision | bf16 | Faster training, lower memory |

Expect ~2-3 hours on A100

The model checkpoint and LoRA weights will be saved to `./outputs_visual_jigsaw/`.

In [ ]:
# Full-data one-epoch run
train_dataset = VisualReasoningDataset(split["train"], processor)
eval_dataset  = VisualReasoningDataset(split["test"].select(range(min(256, len(split["test"])))), processor)

print("Train size:", len(train_dataset))
print("Eval size :", len(eval_dataset))

model.config.text_loss_weight = 1.0
model.config.img_loss_weight = 0.1

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

model.config.use_cache = False

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    warmup_steps=20,
    logging_steps=20,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=2,
    label_names=["labels"],
    gradient_checkpointing=False,
    eval_strategy="no",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

train_result = trainer.train()
print(train_result)

trainer.save_model(OUTPUT_DIR)
latent = model.get_model().latent_queries.detach().cpu()
torch.save({"latent_queries": latent}, os.path.join(OUTPUT_DIR, "latent_queries.pt"))
print(f"Training complete. Model saved to {OUTPUT_DIR}")

Train size: 17519
Eval size : 256


Step,Training Loss
20,74.215400
40,73.051700
60,73.284700
80,69.921500
100,68.856500
120,68.805700
140,65.283300
160,62.040900
180,62.603200
200,59.966400


TrainOutput(global_step=2189, training_loss=50.79135859573525, metrics={'train_runtime': 5229.0577, 'train_samples_per_second': 3.35, 'train_steps_per_second': 0.419, 'total_flos': 6.002255702801996e+17, 'train_loss': 50.79135859573525, 'epoch': 0.9996004338147154})
Training complete. Model saved to ./outputs_visual_jigsaw


## Compare Before vs After

Let's see the impact of fine-tuning!

In [ ]:
import os
import numpy as np
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

def seeded_eval_losses(model, batch, seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return eval_losses(model, batch)

n_eval = min(64, len(eval_dataset))
indices = list(range(n_eval))
seeds = [0, 1, 2]

initial_state = torch.load(os.path.join(OUTPUT_DIR, "initial_trainable_state.pt"), map_location="cpu")
ft_state = get_trainable_state(model)

base_total, ft_total = [], []
base_text, ft_text = [], []
base_img, ft_img = [], []

improved = 0
worsened = 0
flat = 0

for idx in indices:
    batch = move_batch_to_cuda(collate_fn([eval_dataset[idx]]))

    base_runs = []
    ft_runs = []

    for seed in seeds:
        load_trainable_state(model, initial_state)
        base_vals = seeded_eval_losses(model, batch, seed)
        base_runs.append(base_vals)

        load_trainable_state(model, ft_state)
        ft_vals = seeded_eval_losses(model, batch, seed)
        ft_runs.append(ft_vals)

    base_loss = np.mean([x["loss"] for x in base_runs])
    ft_loss   = np.mean([x["loss"] for x in ft_runs])

    base_t = np.mean([x["text_loss"] for x in base_runs])
    ft_t   = np.mean([x["text_loss"] for x in ft_runs])

    base_i = np.mean([x["img_loss"] for x in base_runs])
    ft_i   = np.mean([x["img_loss"] for x in ft_runs])

    base_total.append(base_loss)
    ft_total.append(ft_loss)
    base_text.append(base_t)
    ft_text.append(ft_t)
    base_img.append(base_i)
    ft_img.append(ft_i)

    delta = ft_loss - base_loss
    if delta < -1e-4:
        improved += 1
    elif delta > 1e-4:
        worsened += 1
    else:
        flat += 1

print("Base avg loss   :", float(np.mean(base_total)))
print("FT avg loss     :", float(np.mean(ft_total)))
print("Avg total delta :", float(np.mean(ft_total) - np.mean(base_total)))
print("Avg text delta  :", float(np.mean(ft_text) - np.mean(base_text)))
print("Avg image delta :", float(np.mean(ft_img) - np.mean(base_img)))
print()
print("Improved samples:", improved)
print("Worsened samples:", worsened)
print("Flat samples    :", flat)

Base avg loss   : 9.086186123390991
FT avg loss     : 5.9237147619326915
Avg total delta : -3.1624713614582998
Avg text delta  : -3.129117973148823
Avg image delta : -0.33353377444048715

Improved samples: 64
Worsened samples: 0
Flat samples    : 0


What did we learn? After stabilizing the BLIP3o training path and evaluating on a 64-example held-out slice with repeated seeded loss estimates, the fine-tuned adapter reduced average loss from 9.09 to 5.92, improving all 64 evaluated examples relative to the adapter’s initialization baseline. Most of the gain came from lower text loss, with a smaller but still positive improvement in image loss.